# quant-kit — GGUF Quantization Pipeline
> **GitHub**: [DhruvalPtl/quant-kit](https://github.com/DhruvalPtl/quant-kit) | **HF**: [Dhptl](https://huggingface.co/Dhptl)

## Before you start
1. `Runtime → Change runtime type → T4 GPU`
2. Add HF token to Colab Secrets (🔑 left sidebar): **Name** = `HF_TOKEN`

## Two workflows
| First time (night) | Resume (morning) |
|---|---|
| Cells 1 → 2 → 3 → 4 → 5 → 6 | Cells 1 → 2 → 3 → 4 → 7 → 8 → 9 → 10 |
| Cell 6 saves GGUFs to Drive | Cell 7 restores GGUFs from Drive |


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 1 — Runtime check (GPU / Disk / RAM)
# ══════════════════════════════════════════════════════════════
import subprocess, shutil, psutil

gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader',
                     shell=True, capture_output=True, text=True)
print(f'[OK] GPU  : {gpu.stdout.strip()}' if gpu.returncode == 0
      else '[!!] No GPU! Go to Runtime > Change runtime type > T4 GPU')

disk = shutil.disk_usage('/')
print(f'[OK] Disk : {disk.free/1e9:.1f} GB free of {disk.total/1e9:.1f} GB')

ram = psutil.virtual_memory()
print(f'[OK] RAM  : {ram.available/1e9:.1f} GB available of {ram.total/1e9:.1f} GB')

if disk.free/1e9 < 50:
    print('[!!] Less than 50 GB free — may be tight for 12B model')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 2 — Clone quant-kit & install llama.cpp
# Set FORCE_REINSTALL = True if Cell 3 shows llama-quantize errors
# ══════════════════════════════════════════════════════════════
import os, shutil, subprocess as _sp
from pathlib import Path

FORCE_REINSTALL = False   # <-- set True if Cell 3 fails

REPO    = 'https://github.com/DhruvalPtl/quant-kit.git'
WORKDIR = '/content/quant-kit'

# Clone or pull latest code
if os.path.exists(WORKDIR):
    print('[OK] quant-kit already cloned — pulling latest...')
    os.system(f'git -C {WORKDIR} pull')
else:
    print('[->] Cloning quant-kit...')
    os.system(f'git clone {REPO} {WORKDIR}')

os.chdir(WORKDIR)
print(f'[OK] Working directory: {os.getcwd()}')

LLAMA_CPP = Path(WORKDIR) / 'llama.cpp'

# Manual force reinstall
if FORCE_REINSTALL and LLAMA_CPP.exists():
    print('[->] Force reinstall: deleting existing llama.cpp/ ...')
    shutil.rmtree(str(LLAMA_CPP))
    print('[OK] Deleted')

# Auto-detect: try running llama-quantize — if it fails, reinstall
# NOTE: new llama.cpp uses small launcher stubs (~17KB) + .so libs, not big monolithic binaries
qbin = LLAMA_CPP / 'llama-quantize'
if qbin.exists() and not FORCE_REINSTALL:
    _env = {**os.environ, 'LD_LIBRARY_PATH': str(LLAMA_CPP)}
    _r = _sp.run([str(qbin), '--help'], capture_output=True, env=_env)
    if _r.returncode != 0:
        print(f'[!!] llama-quantize ({qbin.stat().st_size} bytes) fails to run — auto-reinstalling...')
        shutil.rmtree(str(LLAMA_CPP))
        print('[OK] Removed broken llama.cpp/ — will re-download')
    else:
        print(f'[OK] Existing llama-quantize works ({qbin.stat().st_size} bytes)')

print()
!python setup_linux.py

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 3 — Verify llama-quantize + fix .so symlinks
# ══════════════════════════════════════════════════════════════
import os, re, subprocess
from pathlib import Path
from collections import defaultdict

LLAMA_CPP = Path('/content/quant-kit/llama.cpp')

# Fix 1: libXXX.so (file) missing libXXX.so.0
for f in list(LLAMA_CPP.glob('*.so')):
    if not f.is_symlink():
        v = LLAMA_CPP / (f.name + '.0')
        if not v.exists():
            os.symlink(f.name, str(v))

# Fix 2: libXXX.so.0.0.NNNN (file) missing libXXX.so.0 symlink
three_part = re.compile(r'^(lib.+\.so)\.(\d+)\.\d+\.\d+$')
by_base = defaultdict(list)
for f in LLAMA_CPP.glob('lib*.so.*'):
    m = three_part.match(f.name)
    if m and not f.is_symlink():
        by_base[m.group(1)].append((int(m.group(2)), f.name))

fixed = 0
for base_so, versions in by_base.items():
    versions.sort(reverse=True)
    latest, major = versions[0][1], versions[0][0]
    so_major = LLAMA_CPP / f'{base_so}.{major}'
    if not so_major.exists():
        os.symlink(latest, str(so_major))
        fixed += 1

if fixed:
    print(f'[OK] Created {fixed} missing .so versioned symlinks')

# Test llama-quantize
# NOTE: new llama.cpp launcher stubs are small (~17KB) — size check is NOT valid
env  = {**os.environ, 'LD_LIBRARY_PATH': str(LLAMA_CPP)}
qbin = LLAMA_CPP / 'llama-quantize'

if not qbin.exists():
    print('[ERR] llama-quantize not found — re-run Cell 2 with FORCE_REINSTALL = True')
else:
    print(f'[->] Testing llama-quantize ({qbin.stat().st_size} bytes)...')
    result = subprocess.run([str(qbin), '--help'], capture_output=True, text=True, env=env)
    if result.returncode == 0:
        print('[OK] llama-quantize works!')
        print('[OK] Ready to proceed.')
    else:
        print(f'[ERR] llama-quantize failed (exit {result.returncode})')
        print(result.stderr[:400])
        ldd = subprocess.run(['ldd', str(qbin)], capture_output=True, text=True, env=env)
        missing = [l.strip() for l in ldd.stdout.splitlines() if 'not found' in l]
        if missing:
            print('\nMissing libraries:')
            for m in missing:
                print(f'  {m}')
        print('\nFix: Re-run Cell 2 with FORCE_REINSTALL = True')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 4 — HuggingFace authentication
# ══════════════════════════════════════════════════════════════
import os
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    with open('/content/quant-kit/.env', 'w') as f:
        f.write(f'hf_token = "{hf_token}"\n')
    from huggingface_hub import HfApi
    user = HfApi(token=hf_token).whoami()
    print(f'[OK] Logged in as: {user["name"]}')
except Exception as e:
    print(f'[ERR] {e}')
    print('  1. Click 🔑 in left sidebar')
    print('  2. Add secret: Name=HF_TOKEN, Value=your_token')
    print('  3. Toggle "Notebook access" ON')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 5 — Quantize  [FIRST TIME ONLY — skip on resume]
# ══════════════════════════════════════════════════════════════
import os
from pathlib import Path

MODEL_ID = 'google/gemma-4-12B'           # <-- change for different models
QUANTS   = 'Q4_K_M Q5_K_M Q8_0 IQ4_XS'

os.chdir('/content/quant-kit')
MODEL_NAME = MODEL_ID.split('/')[-1]

# Auto-skip if GGUFs already exist
output_dir = Path(f'/content/quant-kit/output/{MODEL_NAME}')
existing   = list(output_dir.glob('*.gguf')) if output_dir.exists() else []
if existing:
    print('[OK] GGUFs already exist — skipping quantize')
    for f in sorted(existing):
        print(f'  {f.name}  ({f.stat().st_size/1e9:.1f} GB)')
    print('\nProceed to Cell 6 (Save) or Cell 8 (Benchmark)')
else:
    !python quantize.py \
        --model {MODEL_ID} \
        --quants {QUANTS} \
        --delete-src

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 6 — Save GGUFs to Google Drive  [Run BEFORE sleep]
# ══════════════════════════════════════════════════════════════
from google.colab import drive
import shutil
from pathlib import Path

MODEL_NAME = 'gemma-4-12B'   # <-- match your model

drive.mount('/content/drive')

SRC  = Path(f'/content/quant-kit/output/{MODEL_NAME}')
DEST = Path(f'/content/drive/MyDrive/quant-kit-output/{MODEL_NAME}')
DEST.mkdir(parents=True, exist_ok=True)

files = sorted(SRC.glob('*.gguf'))
print(f'Saving {len(files)} files to Drive...')
for f in files:
    dest_file = DEST / f.name
    if dest_file.exists():
        print(f'  [skip] {f.name} — already in Drive')
    else:
        print(f'  Copying {f.name} ({f.stat().st_size/1e9:.1f} GB)...')
        shutil.copy2(str(f), str(dest_file))
        print(f'  [OK] Done')

print('\nFiles saved to Drive:')
for f in sorted(DEST.glob('*.gguf')):
    print(f'  {f.name}  ({f.stat().st_size/1e9:.1f} GB)')
print('\nSleep well! Resume tomorrow: Cells 1 → 2 → 3 → 4 → 7 → 8 → 9 → 10')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 7 — Restore GGUFs from Drive  [Morning RESUME]
# Run instead of Cell 5 when resuming from saved session
# ══════════════════════════════════════════════════════════════
from google.colab import drive
import shutil
from pathlib import Path

MODEL_NAME = 'gemma-4-12B'   # <-- match your model

drive.mount('/content/drive')

SRC  = Path(f'/content/drive/MyDrive/quant-kit-output/{MODEL_NAME}')
DEST = Path(f'/content/quant-kit/output/{MODEL_NAME}')
DEST.mkdir(parents=True, exist_ok=True)

if not SRC.exists() or not list(SRC.glob('*.gguf')):
    print('[ERR] No GGUFs found in Drive!')
    print(f'      Expected path: {SRC}')
    print('      Run Cell 5 first to quantize.')
else:
    files = sorted(SRC.glob('*.gguf'))
    print(f'Restoring {len(files)} GGUFs from Drive...')
    for f in files:
        dest_file = DEST / f.name
        if dest_file.exists():
            print(f'  [skip] {f.name} — already local')
        else:
            print(f'  Restoring {f.name} ({f.stat().st_size/1e9:.1f} GB)...')
            shutil.copy2(str(f), str(dest_file))
            print(f'  [OK] Done')

    print('\nFiles ready locally:')
    for f in sorted(DEST.glob('*.gguf')):
        print(f'  {f.name}  ({f.stat().st_size/1e9:.1f} GB)')
    print('\n[OK] Run Cell 8 (benchmark)')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 8 — Benchmark
# ══════════════════════════════════════════════════════════════
import os
os.chdir('/content/quant-kit')

MODEL_NAME = 'gemma-4-12B'

# ngl=99 offloads all layers to GPU (T4 = 15GB VRAM)
# ngl=0  for CPU-only if model doesn't fit in VRAM
!python benchmark.py --model {MODEL_NAME} --ngl 99

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 9 — Generate model card (README.md)
# ══════════════════════════════════════════════════════════════
import os
os.chdir('/content/quant-kit')

MODEL_NAME = 'gemma-4-12B'
MODEL_ID   = 'google/gemma-4-12B'
HF_AUTHOR  = 'Dhptl'

!python model_card.py \
    --model {MODEL_NAME} \
    --original {MODEL_ID} \
    --author {HF_AUTHOR}

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 10 — Upload to HuggingFace
# Creates Dhptl/gemma-4-12B-GGUF automatically
# ══════════════════════════════════════════════════════════════
import os
os.chdir('/content/quant-kit')

MODEL_NAME = 'gemma-4-12B'
HF_AUTHOR  = 'Dhptl'

!python upload.py \
    --model {MODEL_NAME} \
    --author {HF_AUTHOR}

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 11 — Cleanup local files (run after upload)
# ══════════════════════════════════════════════════════════════
import shutil, os
from pathlib import Path

MODEL_NAME = 'gemma-4-12B'

output = Path(f'/content/quant-kit/output/{MODEL_NAME}')
if output.exists():
    shutil.rmtree(str(output))
    print(f'[OK] Cleaned: {output}')

disk = shutil.disk_usage('/')
print(f'[OK] Disk after cleanup: {disk.free/1e9:.1f} GB free')
print()
print('To quantize another model:')
print('  1. Change MODEL_ID in Cell 5')
print('  2. Run Cells 5 → 6 → 8 → 9 → 10 → 11')